<a href="https://colab.research.google.com/github/matthewpecsok/IS4490_student_course_files/blob/main/module5-inclass-demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/matthewpecsok/IS4490-creation-fall2026/blob/main/module-05-ai-augmented-workflows/module-05-assignment-05-prompt-tool-workflow-template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 5 Assignment 5: Weather Data Retrieval and Export Workflow

**Notebook:** Student Build Template
**Runtime:** Jupyter or Google Colab with Ollama and LangChain
**Model:** `gemma4:12b` through Ollama

This notebook demonstrates a workflow for retrieving current weather information for a user-specified city and exporting it to JSON and CSV files. You will build three tools for this purpose:

1. Retrieve weather data (`get_weather_data`)
2. Write data to JSON (`write_to_json`)
3. Write data to CSV (`write_to_csv`)

Each tool's result is remembered and fed into the next step, so the conversation feels continuous even though every model call is stateless underneath. That "memory" is plumbing the notebook provides for you. The three tools are what you build.

This is still not an agent. The **order** of the three steps is fixed by the notebook. The model only decides, at each fixed step, whether the user's question requires calling that step's tool and with what arguments.

## Important Instructions

1. Read the Module 5 assignment before editing this notebook.
2. Complete the cells marked `TODO`.
3. Use only the fictional data created in this notebook.
4. Do not connect to live CRM, sales, finance, email, or customer-communication systems.
5. Leave visible evidence of your tool tests, chat steps, failures, revisions, and final test results.
6. The language model may draft or explain text. Deterministic code must handle field exclusion, no-match behavior, arithmetic, the validation checkpoint, and the disclaimer.
7. Several functions intentionally raise `NotImplementedError`. Replace those placeholders with your own code before running the checkpoints.


---
## Setup: Installing Ollama and LangChain

If you are using Google Colab, run the setup cells below. If you are running locally and already have Ollama installed and running, you may skip the install cells and start at the imports cell.

This notebook uses `gemma4:12b` because it supports tool calling through LangChain.

In [1]:
# Colab setup step 1: install zstd, which the Ollama installer expects on Ubuntu.
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 52 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (527 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [2]:
# Colab setup step 2: install Ollama.
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [120]:
# Colab setup step 3: start the Ollama server and pull the model.
import subprocess
import time

ollama_server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

pull = subprocess.run(["ollama", "pull", "gemma4:12b"], capture_output=True, text=True)
print(pull.stdout[-1000:])
print(pull.stderr[-1000:])
print("Ollama setup attempted. If the pull failed, check the runtime log and rerun this cell.")


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 1278394b6936: 100% ▕██████████████████▏ 7.4 GB                         
pulling 675ad6e68101: 100% ▕██████████████████▏ 175 MB                         
pulling 0d542e0c8804: 100% ▕██████████████████▏  10 KB                         
pulling 56380ca2ab89: 100% ▕██████████████████▏   42 B                         
pulling c805f5b265d8: 100% ▕██████████████████▏  548 B                         
verifying sha256 digest 
writing manifest 
success 

Ollama setup attempted. If the pull failed, check the runtime log and rerun this cell.


In [121]:
# Colab setup step 4: install Python packages.
!pip install langchain langchain-ollama --quiet

In [122]:
from datetime import datetime

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

MODEL = "gemma4:12b"
llm = ChatOllama(model=MODEL, temperature=0)

print(f"Ready. Using {MODEL} through Ollama and LangChain.")

Ready. Using gemma4:12b through Ollama and LangChain.


In [123]:
def require_finished(label, value):
    """Stop before a run when required student work is unfinished."""
    if value is None or "TODO" in str(value):
        raise ValueError(f"Complete {label} before running this cell.")


def pretty(obj):
    import json
    print(json.dumps(obj, indent=2, sort_keys=True))


def money(value):
    return f"${float(value):,.2f}"

---
## Skill and Workflow Scenario: Weather Data Retrieval and Export

The goal is to retrieve current weather data for a city and export it. The **Weather Data Retrieval and Export Skill** performs these actions:

1. Retrieve weather data for a specified city.
2. Write the retrieved weather data to a JSON file.
3. Write the retrieved weather data to a CSV file.

The skill retrieves and exports weather data only. It never modifies external weather data sources or makes any decisions based on the data. A person must review the generated files.

---
## Build Map From Earlier Modules

| Earlier pattern | Where to reuse it in Module 5 |
|---|---|
| Module 2 prompt calls | Build `SystemMessage` and `HumanMessage`, call `llm.invoke`, inspect the response. |
| Module 3 repeatable runs | Keep run evidence, compare expected vs. actual results, revise and rerun after a failure. |
| Module 4 tool categories | Use `@tool`, strong docstrings, typed inputs, direct `.invoke()` tests, controlled no-match behavior. |

The new work in Module 5 is chaining: letting one tool's output become the next step's context, and adding one deterministic checkpoint before AI-drafted content is allowed to leave the notebook as a file.


---
## Skill Definition

Before building the tools, define the business skill your four tools implement together. This is a design artifact — Ollama does not create a native "skill" object for you.


In [124]:
sales_meeting_prep_skill = {
    "skill_name": "Weather Data Retrieval and Export Skill",
    "business_goal": "Retrieve current weather information for a user-specified city using an API and export it to JSON and CSV files.",
    "trigger_or_input": "A city name for which to get weather data.",
    "final_output": "JSON and CSV files containing the retrieved weather data.",
    "tools_used_in_order": [
        "get_weather_data",
        "write_to_json",
        "write_to_csv"
    ],
    "what_the_model_may_decide": [
        "which tool to call at each fixed step, and with what arguments"
    ],
    "what_the_model_may_never_decide": [
        "the order of the three steps",
        "the format of the output files (JSON/CSV)",
        "whether to include specific fields in the output"
    ],
    "human_responsibilities": [
        "Review the retrieved weather data for accuracy",
        "Verify the content of the generated JSON and CSV files"
    ],
    "prohibited_actions": [
        "Accessing any API not explicitly provided by the user",
        "Modifying external weather data sources",
        "Accessing protected or private user data"
    ]
}

pretty(sales_meeting_prep_skill)

{
  "business_goal": "Retrieve current weather information for a user-specified city using an API and export it to JSON and CSV files.",
  "final_output": "JSON and CSV files containing the retrieved weather data.",
  "human_responsibilities": [
    "Review the retrieved weather data for accuracy",
    "Verify the content of the generated JSON and CSV files"
  ],
  "prohibited_actions": [
    "Accessing any API not explicitly provided by the user",
    "Modifying external weather data sources",
    "Accessing protected or private user data"
  ],
  "skill_name": "Weather Data Retrieval and Export Skill",
  "tools_used_in_order": [
    "get_weather_data",
    "write_to_json",
    "write_to_csv"
  ],
  "trigger_or_input": "A city name for which to get weather data.",
  "what_the_model_may_decide": [
    "which tool to call at each fixed step, and with what arguments"
  ],
  "what_the_model_may_never_decide": [
    "the order of the three steps",
    "the format of the output files (JSON/C

## Responsible AI and Company-Data Check

Complete this check before building the tools. It forces you to define what your tools are allowed to expose and what stays human responsibility.

In [125]:
company_data_responsibility_check = {
    "data_source": "Free Weather API (e.g., OpenWeatherMap)",
    "allowed_fields": [
        "City Name",
        "Temperature (Celsius)",
        "Weather Description",
        "Humidity",
        "Wind Speed"
    ],
    "excluded_fields": [
        "None (all extracted fields are intended for output)"
    ],
    "authorized_user": "Any user needing to retrieve and store weather information.",
    "purpose_limit": "To retrieve current weather data for a specified city for analytical or informational purposes.",
    "human_review_trigger": "Before any use or distribution of the generated JSON/CSV files, a human must review the content for accuracy and completeness.",
    "logging_need": "Record the city queried, the date of retrieval, and the filenames of generated outputs."
}

pretty(company_data_responsibility_check)

{
  "allowed_fields": [
    "City Name",
    "Temperature (Celsius)",
    "Weather Description",
    "Humidity",
    "Wind Speed"
  ],
  "authorized_user": "Any user needing to retrieve and store weather information.",
  "data_source": "Free Weather API (e.g., OpenWeatherMap)",
  "excluded_fields": [
    "None (all extracted fields are intended for output)"
  ],
  "human_review_trigger": "Before any use or distribution of the generated JSON/CSV files, a human must review the content for accuracy and completeness.",
  "logging_need": "Record the city queried, the date of retrieval, and the filenames of generated outputs.",
  "purpose_limit": "To retrieve current weather data for a specified city for analytical or informational purposes."
}


---
## Part 1: Data Handling Tools

This section describes the generic tools for data handling (writing to JSON and CSV), which are independent of the data source. While the `get_weather_data` tool retrieves the data, `write_to_json` and `write_to_csv` handle its storage.

In [126]:
# CRM Data setup removed as it is not applicable to the new task.
# This notebook now focuses on web scraping and data export.

---
## A Small Helper: Chatbot Memory Across Steps

This is the plumbing behind the chatbot. It is provided for you so you can study it, not build it.

Every call to the local model is stateless -- the model does not remember earlier turns on its own. `ask_chatbot_step` sends one chat turn: it takes the rep's question, an optional block of context text, and the one tool available at this step. It prints what happened and returns a small dictionary describing it -- which tool was called, with what arguments, what the tool returned, and the model's final answer.

`ask_chatbot_step` does **not** save anything to `RESEARCH_TRAIL` by itself. That is deliberate. The saving happens in the notebook cell that calls it, one explicit line at a time, so the "memory" is something you can actually see happening in your own code instead of something hidden inside a helper function.

The pattern you will repeat at each step looks like this:

```python
step1 = ask_chatbot_step(question, [some_tool], context_text=format_research_trail())

RESEARCH_TRAIL["some_key"] = step1["tool_result"]   # <-- this line is the chaining
```

`format_research_trail()` is a second small helper. It just reads whatever is currently in `RESEARCH_TRAIL` and turns it into readable text so it can be dropped into the next step's system prompt. It does not write anything -- only your own `RESEARCH_TRAIL[...] = ...` lines do that.


In [127]:
RESEARCH_TRAIL = {
    "weather_data": None,
    "json_file_status": None,
    "csv_file_status": None
}

def reset_research_trail():
    for key in RESEARCH_TRAIL:
        RESEARCH_TRAIL[key] = None


def format_research_trail():
    """Turn whatever is currently in RESEARCH_TRAIL into readable text.
    This only reads RESEARCH_TRAIL -- it never writes to it.
    """
    lines = []
    for key, value in RESEARCH_TRAIL.items():
        if value is not None:
            if isinstance(value, (list, dict)):
                # Convert Python objects to JSON string for context
                lines.append(f"{key}: {json.dumps(value, indent=2)}")
            else:
                lines.append(f"{key}: {value}")
    return "\n".join(lines) if lines else "No research has been gathered yet this session."


def ask_chatbot_step(user_message, tools, context_text=None):
    """Send one chat turn to the model and let it decide whether to call the
    provided tool.

    This function does NOT save anything to RESEARCH_TRAIL. It just runs one
    turn and hands back what happened: which tool was called, with what
    arguments, what the tool returned, and the model's final answer. The cell
    that calls this function decides what is worth remembering and writes it
    into RESEARCH_TRAIL itself -- watch for that line in each step below.
    """
    system_prompt = (
        "You are a data extraction assistant. Only call a tool when the "
        "user's question actually requires it. Use the previously extracted "
        "data stored in RESEARCH_TRAIL as context when performing subsequent operations."
    )
    if context_text:
        system_prompt += f"\n\nResearch gathered so far:\n{context_text}"

    tool_map = {t.name: t for t in tools}
    llm_with_tools = llm.bind_tools(tools)

    messages = [SystemMessage(content=system_prompt), HumanMessage(content=user_message)]
    response = llm_with_tools.invoke(messages)

    print(f"User: {user_message}")

    if not response.tool_calls:
        print("Tool called: none")
        print(f"Assistant: {response.content}\n")
        return {"tool_called": None, "arguments": None, "tool_result": None, "final_answer": response.content}

    call = response.tool_calls[0]
    tool_obj = tool_map[call["name"]]
    result = tool_obj.invoke(call["args"])

    print(f"Tool called: {call['name']}")
    print(f"Arguments: {call['args']}")
    # The result itself is a Python object, print its structured form if possible
    if isinstance(result, (list, dict)):
        print(f"Tool result: {json.dumps(result, indent=2)}")
    else:
        print(f"Tool result: {result}")

    from langchain_core.messages import ToolMessage
    messages.append(response)
    # Ensure ToolMessage content is a JSON string if the result is a Python object
    tool_message_content = json.dumps(result, indent=2) if isinstance(result, (list, dict)) else str(result)
    messages.append(ToolMessage(content=tool_message_content, tool_call_id=call["id"], name=call["name"]))
    final = llm_with_tools.invoke(messages)
    print(f"Assistant: {final.content}\n")

    return {
        "tool_called": call["name"],
        "arguments": call["args"],
        "tool_result": result,
        "final_answer": final.content
    }

---
## Step 1: Get Weather Data

**Business problem:** The primary goal is to obtain current weather conditions for a specified city. This tool simulates fetching data from a weather API.

In [128]:
import requests
from bs4 import BeautifulSoup
import json
import random
from typing import List, Dict, Any

@tool
def get_weather_data(city: str) -> List[Dict[str, Any]]:
    """Fetches current weather information for a specified city.
    This tool simulates fetching data from a weather API. For a real implementation, it would make an HTTP request to a weather service.
    Returns a list of dictionaries containing the city, temperature, description, humidity, and wind speed.
    """
    # Placeholder for a real API key
    # weather_api_key = "YOUR_OPENWEATHERMAP_API_KEY"
    # base_url = "http://api.openweathermap.org/data/2.5/weather?"

    # For demonstration, we'll return simulated data.
    if city.lower() == "unknown city":
        return [{"error": "City not found or invalid city name provided."}]

    # Simulate API call delay
    # import time
    # time.sleep(1)

    # Simulate different weather conditions
    descriptions = ["clear sky", "few clouds", "scattered clouds", "broken clouds", "shower rain", "rain", "thunderstorm", "snow", "mist"]
    temperatures = [round(random.uniform(0.0, 30.0), 1), round(random.uniform(-10.0, 15.0), 1), round(random.uniform(15.0, 35.0), 1)]
    humidity = random.randint(30, 100)
    wind_speed = round(random.uniform(0.5, 15.0), 1)

    weather_info = {
        "City Name": city.title(),
        "Temperature (Celsius)": random.choice(temperatures),
        "Weather Description": random.choice(descriptions),
        "Humidity": humidity,
        "Wind Speed": wind_speed
    }

    return [weather_info]

### Direct Tool Tests

Run the tool directly before placing it inside a chat step.

In [129]:
# After implementing the tool, run these direct tests and leave the outputs visible.
print(get_weather_data.invoke({"city": "London"}))
print()
print(get_weather_data.invoke({"city": "New York"}))
print()
print(get_weather_data.invoke({"city": "Unknown City"}))

[{'City Name': 'London', 'Temperature (Celsius)': -5.3, 'Weather Description': 'rain', 'Humidity': 70, 'Wind Speed': 14.0}]

[{'City Name': 'New York', 'Temperature (Celsius)': 17.9, 'Weather Description': 'thunderstorm', 'Humidity': 56, 'Wind Speed': 0.8}]

[{'error': 'City not found or invalid city name provided.'}]


### Ask the Chatbot: Step 1

The user asks for weather data for a city. The assistant should call `get_weather_data`.

In [130]:
city_to_query = "Salt Lake City"
step1 = ask_chatbot_step(
    f"What is the current weather in {city_to_query}?",
    [get_weather_data],
    context_text=format_research_trail()
)

# This is the chaining step: explicitly save what we learned so Steps 2-4 can use it.
RESEARCH_TRAIL["weather_data"] = step1["tool_result"]

print("RESEARCH_TRAIL now contains:")
pretty(RESEARCH_TRAIL)

User: What is the current weather in Salt Lake City?
Tool called: get_weather_data
Arguments: {'city': 'Salt Lake City'}
Tool result: [
  {
    "City Name": "Salt Lake City",
    "Temperature (Celsius)": 9.6,
    "Weather Description": "clear sky",
    "Humidity": 42,
    "Wind Speed": 2.4
  }
]
Assistant: The current weather in Salt Lake City is clear sky with a temperature of 9.6°C, 42% humidity, and a wind speed of 2.4.

RESEARCH_TRAIL now contains:
{
  "csv_file_status": null,
  "json_file_status": null,
  "weather_data": [
    {
      "City Name": "Salt Lake City",
      "Humidity": 42,
      "Temperature (Celsius)": 9.6,
      "Weather Description": "clear sky",
      "Wind Speed": 2.4
    }
  ]
}


---
## Step 2: Write Data to JSON

**Business problem:** After retrieving the weather data, it needs to be stored in a structured, machine-readable format. This tool writes the data to a JSON file.

In [131]:
import json
from typing import List, Dict, Any

@tool
def write_to_json(data: List[Dict[str, Any]], filename: str) -> str:
    """Writes the provided data (list of dictionaries) to a specified JSON file.
    Use this tool to save structured data.
    """
    try:
        with open(filename, 'w') as f:
            json.dump(data, f, indent=4)
        return f"Successfully wrote data to {filename}"
    except Exception as e:
        return f"Error writing to JSON file {filename}: {e}"

In [132]:
# After implementing the tool, run these direct tests and leave the outputs visible.
# Assuming `weather_data_example` is a valid Python list of dictionaries
weather_data_example = [{"City Name": "London", "Temperature (Celsius)": 15.5, "Weather Description": "few clouds", "Humidity": 70, "Wind Speed": 8.2}]
print(write_to_json.invoke({"data": weather_data_example, "filename": "test_weather.json"}))

Successfully wrote data to test_weather.json


### Ask the Chatbot: Step 2

The assistant should save the weather data obtained in Step 1 to a JSON file.

In [145]:
format_research_trail()

'weather_data: [\n  {\n    "City Name": "Salt Lake City",\n    "Temperature (Celsius)": 9.6,\n    "Weather Description": "clear sky",\n    "Humidity": 42,\n    "Wind Speed": 2.4\n  }\n]'

In [146]:
step2 = ask_chatbot_step(
    "Save the json weather data to a JSON file named 'city_weather.json'.",
    [write_to_json],
    context_text=format_research_trail()
)

RESEARCH_TRAIL["json_file_status"] = step2["tool_result"]

print("RESEARCH_TRAIL now contains:")
pretty(RESEARCH_TRAIL)

User: Save the json weather data to a JSON file named 'city_weather.json'.
Tool called: none
Assistant: 

RESEARCH_TRAIL now contains:
{
  "csv_file_status": null,
  "json_file_status": null,
  "weather_data": [
    {
      "City Name": "Salt Lake City",
      "Humidity": 42,
      "Temperature (Celsius)": 9.6,
      "Weather Description": "clear sky",
      "Wind Speed": 2.4
    }
  ]
}


---
## Step 3: Write Data to CSV

**Business problem:** For broader compatibility and ease of analysis, the weather data should also be available in a tabular format. This tool converts the JSON data to a pandas DataFrame and writes it to a CSV file.

In [134]:
import pandas as pd
import json
from typing import List, Dict, Any

@tool
def write_to_csv(data: List[Dict[str, Any]], filename: str) -> str:
    """Writes the provided data (list of dictionaries)
    to a specified CSV file.
    Use this tool to save tabular data in CSV format.
    """
    try:
        df = pd.DataFrame(data)
        df.to_csv(filename, index=False)
        return f"Successfully wrote data to {filename}"
    except Exception as e:
        return f"Error writing to CSV file {filename}: {e}"

In [135]:
# After implementing the tool, run these direct tests and leave the outputs visible.
# Assuming `weather_data_example` is a valid Python list of dictionaries
weather_data_example = [{"City Name": "London", "Temperature (Celsius)": 15.5, "Weather Description": "few clouds", "Humidity": 70, "Wind Speed": 8.2}]
print(write_to_csv.invoke({"data": weather_data_example, "filename": "test_weather.csv"}))

Successfully wrote data to test_weather.csv


### Ask the Chatbot: Step 3

The assistant should save the weather data to a CSV file.

In [142]:
step3 = ask_chatbot_step(
    "Now, please save the tabular weather data, which is available in RESEARCH_TRAIL['weather_data'], to a CSV file named 'city_weather.csv'.",
    [write_to_csv],
    context_text=format_research_trail()
)

RESEARCH_TRAIL["csv_file_status"] = step3["tool_result"]

print("RESEARCH_TRAIL now contains:")
pretty(RESEARCH_TRAIL)

User: Now, please save the tabular weather data, which is available in RESEARCH_TRAIL['weather_data'], to a CSV file named 'city_weather.csv'.
Tool called: none
Assistant: 

RESEARCH_TRAIL now contains:
{
  "csv_file_status": null,
  "json_file_status": null,
  "weather_data": [
    {
      "City Name": "Salt Lake City",
      "Humidity": 42,
      "Temperature (Celsius)": 9.6,
      "Weather Description": "clear sky",
      "Wind Speed": 2.4
    }
  ]
}


---

In [137]:
# The original Step 4 for writing a meeting brief is not applicable to the new task.
# The workflow now concludes with saving data to JSON and CSV files.

In [138]:
# This cell previously contained the `write_meeting_brief` tool, which has been removed as it is not part of the new weather data workflow.

This cell previously discussed the `write_meeting_brief` tool and its direct tests, which are no longer relevant to the weather data workflow and have been removed.

In [139]:
# This cell previously contained direct tests for `write_meeting_brief`, which have been removed.

This cell previously discussed the `write_meeting_brief` tool and its direct tests, which are no longer relevant to the weather data workflow and have been removed.

In [140]:
# This cell previously contained the `ask_chatbot_step` for `write_meeting_brief`, which has been removed.

---

---

In [141]:
# The `wget` command was used for web scraping and is no longer relevant for the weather API task.